# Import Required Libraries

In [1]:
# Import basic packages
import os
import warnings
warnings.filterwarnings("ignore")
import datetime
import re
from IPython.display import display, Markdown

# Basic ds packages
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from tqdm import tqdm

# decorator packages
from typing import List, Dict
import warnings
warnings.filterwarnings("ignore")

#sklearn packages
from sklearn.base import BaseEstimator, TransformerMixin

# Tensorflow packages
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.layers import (Dense,
                                    Layer,
                                    BatchNormalization)
from tensorflow.keras.callbacks import (EarlyStopping, 
                                        ModelCheckpoint)
from tensorflow.keras.optimizers import (Adam, 
                                         AdamW, 
                                         RMSprop)
from tensorflow.keras.losses import (SparseCategoricalCrossentropy,
                                     CategoricalCrossentropy)


# Tensorflow Text Packages
import nltk
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud, STOPWORDS
from tensorflow.keras.preprocessing.text import Tokenizer


# GenerateTrainingData

#### Step 0: Custom UDFs

In [2]:
class GenerateTrainingSamples:

    def __init__(self, context_size):
        self.context_size = context_size
        self.mid_point = int(self.context_size/2)

    def __call__(self, training_corpus:Dict):
        self.training_corpus = training_corpus

        training_points = []

        for single_context in tqdm(training_corpus.values()):
            context_training_len = len(single_context)-(self.context_size + 1)
            for iter in range(0, context_training_len):
                
                # Indexing values based on context size
                left_end_split = iter+self.mid_point
                right_start_split = left_end_split + 1
                right_end_split = right_start_split + self.mid_point
                
                # Training Points
                training_points.append([single_context[iter:left_end_split] + \
                                        single_context[right_start_split:right_end_split],\
                                        single_context[right_end_split]])
                
        return training_points

#### 1. Reading Corpus

In [3]:
with open("../artifacts/corpus/text_corpus.pkl", "rb") as f:
    lemma_corpus = pickle.load(f)
    print("Done!! Reading the Lemma Corpus")

Done!! Reading the Lemma Corpus


In [4]:
lemma_corpus_subset = {key: lemma_corpus[key] for key in range(0, 100000)}

#### Step 2 : Generate Training Samples

In [22]:
generator = GenerateTrainingSamples(context_size=4)
training_samples = generator(lemma_corpus_subset)

with open("../artifacts/trainingData/training_samples.pkl","wb") as f:
    pickle.dump(training_samples, f)
    print("Done!! writing training samples")

100%|██████████| 100000/100000 [04:39<00:00, 357.24it/s]


Done!! writing training samples


In [5]:
with open("../artifacts/trainingData/training_samples.pkl","rb") as f:
    training_samples = pickle.load(f)

#### Step 3: Generate Vocabulary

In [24]:
vocabulary_corpus = []
for numpy_val  in tqdm(lemma_corpus_subset.values()):
    vocabulary_corpus.extend(numpy_val)

print(f"Get Unique Vocabulary Size : {len(set(vocabulary_corpus))}")
unique_vocabulary_corpus = set(vocabulary_corpus)

100%|██████████| 100000/100000 [00:00<00:00, 1812287.58it/s]


Get Unique Vocabulary Size : 166418


#### Step 4: OneHotEncoding

In [25]:
from sklearn.preprocessing import OneHotEncoder
oe = OneHotEncoder()
oe.fit(np.array(list(unique_vocabulary_corpus)).reshape(-1,1))

OneHotEncoder()

In [27]:
# Save the fitted model
with open("../artifacts/preprocessing/onehot_encoder.pkl", "wb") as f:
    pickle.dump(oe, f)
    print("Done!! Writting the OneHotEncoder")

# Load the fitted model
with open("../artifacts/preprocessing/onehot_encoder.pkl", "rb") as f:
    oe = pickle.load(f)
    print("Done!! Reading the OneHotEncoder")

Done!! Writting the OneHotEncoder
Done!! Reading the OneHotEncoder


#### Step 5: Training Sample with OneHot Encoded Value

In [36]:
vocabulary_corpus

['this',
 'sound',
 'track',
 'wa',
 'beautiful',
 'it',
 'paint',
 'the',
 'senery',
 'in',
 'your',
 'mind',
 'so',
 'well',
 'i',
 'would',
 'recomend',
 'it',
 'even',
 'to',
 'people',
 'who',
 'hate',
 'vid',
 'game',
 'music',
 'i',
 'have',
 'played',
 'the',
 'game',
 'chrono',
 'cross',
 'but',
 'out',
 'of',
 'all',
 'of',
 'the',
 'game',
 'i',
 'have',
 'ever',
 'played',
 'it',
 'ha',
 'the',
 'best',
 'music',
 'it',
 'back',
 'away',
 'from',
 'crude',
 'keyboarding',
 'and',
 'take',
 'a',
 'fresher',
 'step',
 'with',
 'grate',
 'guitar',
 'and',
 'soulful',
 'orchestra',
 'it',
 'would',
 'impress',
 'anyone',
 'who',
 'care',
 'to',
 'listen',
 '_',
 'im',
 'reading',
 'a',
 'lot',
 'of',
 'review',
 'saying',
 'that',
 'this',
 'is',
 'the',
 'best',
 'game',
 'soundtrack',
 'and',
 'i',
 'figured',
 'that',
 'id',
 'write',
 'a',
 'review',
 'to',
 'disagree',
 'a',
 'bit',
 'this',
 'in',
 'my',
 'opinino',
 'is',
 'yasunori',
 'mitsudas',
 'ultimate',
 'masterpi